# TransitPulse: pretrain-then-finetune demand transformer

**Design choice, and why it isn't just "one big model on everything":** our actual product target is paratransit demand (NYC Access-A-Ride, ~93k training rows). The real, event-level mass-transit corpora we've pulled (NYC TLC rideshare ~1.7B trips, Chicago TNP ~300M, MTA bus/subway hourly ridership in the hundreds of millions) dwarf that by four orders of magnitude. Training one flat model on a mixed loss across all of it would let the huge non-paratransit volume dominate every gradient step and wash out the signal that actually matters.

So this notebook is two stages:

- **Stage 1 (pretrain):** one shared temporal transformer learns general transit-demand structure (daily/weekly cycles, holiday effects, weather sensitivity, cross-city variation) from the large real mass-transit corpora, with a dataset-id embedding so the model knows which system it's looking at.
- **Stage 2 (finetune):** the pretrained backbone is adapted to AAR paratransit demand specifically — new zone (borough) and dataset embeddings, small learning rate on the backbone, full learning rate on the new embeddings and heads. This is a real transfer-learning claim for the application ("pretrained on 2B+ real transit events, fine-tuned on real paratransit demand"), not a parameter-count flex.

Both heads: a Poisson mean-rate head (point forecast) and a quantile/pinball head (native 10/50/90 percentile forecast) trained jointly, so the model ships calibrated-ish uncertainty on its own — conformal calibration (as used in realdata/train_real_models.py) can still wrap this for a coverage guarantee, but the quantiles are learned, not just post-hoc.

Protocol matches the rest of the repo: strict temporal holdout (last 90 days), seasonal-naive baseline (same hour, 7 days earlier), always reported side by side. Setup: Kaggle notebook, GPU on (T4/P100), Internet on, Persistence on.

In [ ]:
%pip install -q duckdb

# Config. TEST_MODE trains on 6 months of TLC only for a quick smoke run
# (~20 min end to end); flip off for the full pretrain + finetune run.
TEST_MODE = False
TLC_START, TLC_END = ("2025-01", "2025-06") if TEST_MODE else ("2019-02", "2026-05")
INPUT_HOURS = 336      # two weeks of history per sample
HORIZON = 24           # predict the next day, hourly
VALID_DAYS = 90
QUANTILES = [0.1, 0.5, 0.9]

# Scaled up from the earlier ~1M-param single-dataset version: with a
# multi-billion-row pretraining corpus, a larger backbone is well justified.
D_MODEL, N_HEADS, N_LAYERS = 256, 8, 6
BATCH, PRETRAIN_EPOCHS, LR = 384, 8, 3e-4
FINETUNE_EPOCHS, FINETUNE_LR, BACKBONE_LR_MULT = 15, 1e-3, 0.05

# Optional: attach a Kaggle Dataset named e.g. "transitpulse-real-panels"
# containing realdata/panels/{chicago_tnp_zone_hour.csv.gz,
# mta_bus_route_hour.csv.gz, mta_subway_station_hour.csv.gz,
# aar_hourly_borough.csv.gz, weather_daily.csv} from the repo. If absent,
# pretraining runs on TLC alone (still ~1.7B rows) and the finetune stage
# is skipped with a clear message telling you what to upload.
EXTRA_PANELS_DIR = "/kaggle/input/transitpulse-real-panels"

In [ ]:
import os, time, requests
from pathlib import Path

RAW = Path("/kaggle/working/raw"); RAW.mkdir(exist_ok=True)
CDN = "https://d37ci6vzurychx.cloudfront.net/trip-data"

def months(a, b):
    y, m = map(int, a.split("-")); y2, m2 = map(int, b.split("-"))
    while (y, m) <= (y2, m2):
        yield f"{y:04d}-{m:02d}"
        y, m = (y + 1, 1) if m == 12 else (y, m + 1)

for mo in months(TLC_START, TLC_END):
    dest = RAW / f"fhvhv_tripdata_{mo}.parquet"
    if dest.exists():
        continue
    r = requests.get(f"{CDN}/{dest.name}", stream=True, timeout=300)
    if r.status_code in (403, 404):
        print(mo, "not published, stopping"); break
    with open(dest, "wb") as f:
        for chunk in r.iter_content(1 << 22):
            f.write(chunk)
    print(mo, f"{dest.stat().st_size/1e6:.0f} MB", flush=True)
print("TLC on disk:", sum(f.stat().st_size for f in RAW.glob('*.parquet'))/1e9, "GB")

In [ ]:
import duckdb

con = duckdb.connect()
tlc = con.execute(f"""
    SELECT CAST(pickup_datetime AS DATE) AS date,
           EXTRACT(hour FROM pickup_datetime) AS hour,
           PULocationID AS zone, COUNT(*) AS y
    FROM read_parquet('{RAW.as_posix()}/fhvhv_tripdata_*.parquet',
                      union_by_name=true)
    GROUP BY ALL ORDER BY zone, date, hour
""").df()
tlc["dataset"] = "nyc_tlc"
tlc["zone"] = tlc["zone"].astype(str)
print(f"nyc_tlc: {len(tlc):,} panel rows, {tlc.y.sum():,} trips, "
      f"{tlc.zone.nunique()} zones, {tlc.date.min()} to {tlc.date.max()}")

In [ ]:
import pandas as pd
from pathlib import Path

# Fold in the other real corpora if the optional Kaggle Dataset is attached.
# Each becomes its own `dataset` id sharing the same (date, hour, zone, y)
# shape, so the pretraining loop below is dataset-agnostic.
extra = Path(EXTRA_PANELS_DIR)
pretrain_frames = [tlc]

def load_panel(path, zone_col, value_col, dataset_name):
    df = pd.read_csv(path)
    df = df.rename(columns={zone_col: "zone", value_col: "y"})
    df["zone"] = df["zone"].astype(str)
    df["dataset"] = dataset_name
    return df[["date", "hour", "zone", "y", "dataset"]]

sources = [
    ("chicago_tnp_zone_hour.csv.gz", "area", "trips", "chicago_tnp"),
    ("mta_bus_route_hour.csv.gz", "route", "riders", "mta_bus"),
    ("mta_subway_station_hour.csv.gz", "station_id", "riders", "mta_subway"),
]
for fname, zcol, vcol, dname in sources:
    fpath = extra / fname
    if fpath.exists():
        df = load_panel(fpath, zcol, vcol, dname)
        pretrain_frames.append(df)
        print(f"{dname}: {len(df):,} rows loaded from attached dataset")
    else:
        print(f"{dname}: not found at {fpath} - pretraining without it")

pretrain = pd.concat(pretrain_frames, ignore_index=True)
pretrain["date"] = pd.to_datetime(pretrain["date"])
print(f"\ncombined pretrain corpus: {len(pretrain):,} rows across "
      f"{pretrain.dataset.nunique()} datasets, {pretrain.y.sum():,.0f} total events")

In [ ]:
import numpy as np

def build_matrix(df):
    """Dense [entity, hour-index] matrix on one shared hourly clock.
    entity = (dataset, zone) pair, so numerically colliding zone ids across
    datasets (e.g. TLC zone '4' and Chicago area '4') never mix."""
    df = df.copy()
    df["entity"] = df["dataset"] + "::" + df["zone"]
    df["ts"] = df["date"] + pd.to_timedelta(df["hour"], unit="h")
    clock = pd.date_range(df.ts.min(), df.ts.max(), freq="h")
    t_index = pd.Series(np.arange(len(clock)), index=clock)
    entities = np.sort(df.entity.unique())
    e_index = {e: i for i, e in enumerate(entities)}
    dataset_of = {e_index[e]: e.split("::")[0] for e in entities}

    M = np.zeros((len(entities), len(clock)), dtype=np.float32)
    M[df.entity.map(e_index).to_numpy(), t_index[df.ts].to_numpy()] = \
        df.y.to_numpy(dtype=np.float32)

    hours = clock.hour.to_numpy(); dows = clock.dayofweek.to_numpy()
    CAL = np.stack([np.sin(2*np.pi*hours/24), np.cos(2*np.pi*hours/24),
                    np.sin(2*np.pi*dows/7),  np.cos(2*np.pi*dows/7)],
                   axis=1).astype(np.float32)
    return M, CAL, entities, e_index, dataset_of, clock

M, CAL, entities, e_index, dataset_of, clock = build_matrix(pretrain)
dataset_names = sorted(pretrain.dataset.unique())
ds_index = {d: i for i, d in enumerate(dataset_names)}
entity_dataset_id = np.array([ds_index[dataset_of[i]] for i in range(len(entities))])

valid_start = len(clock) - VALID_DAYS * 24
starts = np.arange(0, len(clock) - INPUT_HOURS - HORIZON, 24)
train_starts = starts[starts + INPUT_HOURS + HORIZON <= valid_start]
valid_starts = starts[starts + INPUT_HOURS >= valid_start]
print(f"entities {len(entities)} across {len(dataset_names)} datasets, hours {len(clock)}, "
      f"train windows {len(train_starts)*len(entities):,}, "
      f"valid windows {len(valid_starts)*len(entities):,}")

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class Windows(Dataset):
    def __init__(self, starts, entity_ids=None):
        ents = entity_ids if entity_ids is not None else range(M.shape[0])
        self.pairs = [(e, s) for e in ents for s in starts]
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, i):
        e, s = self.pairs[i]
        x = M[e, s:s+INPUT_HOURS]
        y = M[e, s+INPUT_HOURS:s+INPUT_HOURS+HORIZON]
        cal = CAL[s:s+INPUT_HOURS]
        feats = np.concatenate([np.log1p(x)[:, None], cal], axis=1)
        return feats, np.int64(e), np.int64(entity_dataset_id[e]), y

class DemandTransformer(nn.Module):
    """Dual-head: Poisson mean rate (point forecast) + quantile pinball
    (native uncertainty). Entity embedding = zone within a dataset; a
    separate dataset embedding lets the shared backbone tell systems apart
    without needing a unified zone vocabulary across cities."""
    def __init__(self, n_entities, n_datasets, n_quantiles):
        super().__init__()
        self.proj = nn.Linear(5, D_MODEL)
        self.entity_emb = nn.Embedding(n_entities, D_MODEL)
        self.dataset_emb = nn.Embedding(n_datasets, D_MODEL)
        self.pos = nn.Parameter(torch.randn(1, INPUT_HOURS, D_MODEL) * 0.02)
        layer = nn.TransformerEncoderLayer(
            D_MODEL, N_HEADS, 4*D_MODEL, dropout=0.1,
            batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, N_LAYERS)
        self.pool_norm = nn.LayerNorm(D_MODEL)
        self.mean_head = nn.Linear(D_MODEL, HORIZON)
        self.quantile_head = nn.Linear(D_MODEL, HORIZON * n_quantiles)
        self.n_quantiles = n_quantiles

    def forward(self, feats, entity_id, dataset_id):
        h = (self.proj(feats) + self.pos
             + self.entity_emb(entity_id)[:, None, :]
             + self.dataset_emb(dataset_id)[:, None, :])
        h = self.pool_norm(self.enc(h).mean(dim=1))
        mean_rate = nn.functional.softplus(self.mean_head(h)) + 1e-6
        q = self.quantile_head(h).view(-1, HORIZON, self.n_quantiles)
        q = mean_rate.unsqueeze(-1) * torch.exp(q.clamp(-3, 3))  # positive, mean-anchored
        q, _ = torch.sort(q, dim=-1)  # enforce non-crossing quantiles
        return mean_rate, q

def pinball_loss(q_pred, y, quantiles):
    y = y.unsqueeze(-1)
    errors = y - q_pred
    qs = torch.tensor(quantiles, device=y.device).view(1, 1, -1)
    return torch.maximum(qs * errors, (qs - 1) * errors).mean()

dev = "cuda" if torch.cuda.is_available() else "cpu"
model = DemandTransformer(len(entities), len(dataset_names), len(QUANTILES)).to(dev)
print(f"{sum(p.numel() for p in model.parameters())/1e6:.1f}M params, device: {dev}")

## Stage 1: pretrain on the mass-transit corpus

Joint loss: Poisson NLL on the mean-rate head plus pinball loss on the quantile head. Every entity (dataset, zone) is sampled uniformly, so a small dataset (fewer zones) isn't starved relative to TLC's 265 zones.

In [ ]:
train_dl = DataLoader(Windows(train_starts), batch_size=BATCH, shuffle=True,
                      num_workers=2, pin_memory=True, drop_last=True)
valid_dl = DataLoader(Windows(valid_starts), batch_size=BATCH,
                      num_workers=2, pin_memory=True)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, PRETRAIN_EPOCHS*len(train_dl))
scaler = torch.amp.GradScaler()
nll = nn.PoissonNLLLoss(log_input=False, full=False)

best = float("inf")
for ep in range(PRETRAIN_EPOCHS):
    model.train(); t0 = time.time(); tot = 0.0
    for feats, e_id, d_id, y in train_dl:
        feats, e_id, d_id, y = feats.to(dev), e_id.to(dev), d_id.to(dev), y.to(dev)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(dev):
            mean_rate, q = model(feats, e_id, d_id)
            loss = nll(mean_rate, y) + 0.5 * pinball_loss(q, y, QUANTILES)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update(); sched.step()
        tot += loss.item() * len(e_id)
    model.eval(); vtot, n = 0.0, 0
    with torch.no_grad():
        for feats, e_id, d_id, y in valid_dl:
            feats, e_id, d_id, y = feats.to(dev), e_id.to(dev), d_id.to(dev), y.to(dev)
            mean_rate, q = model(feats, e_id, d_id)
            vtot += (nll(mean_rate, y) + 0.5*pinball_loss(q, y, QUANTILES)).item() * len(e_id)
            n += len(e_id)
    v = vtot / n
    print(f"pretrain epoch {ep}: train {tot/len(train_dl.dataset):.4f} "
          f"valid {v:.4f} ({time.time()-t0:.0f}s)", flush=True)
    if v < best:
        best = v
        torch.save(model.state_dict(), "/kaggle/working/pretrained_backbone.pt")
        torch.save({"entities": entities, "dataset_names": dataset_names},
                   "/kaggle/working/pretrain_vocab.pt")

In [ ]:
# Pretrain evaluation vs seasonal-naive (same hour, 7 days earlier)
model.load_state_dict(torch.load("/kaggle/working/pretrained_backbone.pt"))
model.eval()
preds, actuals, naives = [], [], []
with torch.no_grad():
    for feats, e_id, d_id, y in valid_dl:
        mean_rate, _ = model(feats.to(dev), e_id.to(dev), d_id.to(dev))
        preds.append(mean_rate.cpu().numpy()); actuals.append(y.numpy())
        x = np.expm1(feats[:, :, 0].numpy())
        naives.append(x[:, INPUT_HOURS-168:INPUT_HOURS-168+HORIZON])
pred = np.concatenate(preds).ravel(); act = np.concatenate(actuals).ravel()
nai = np.concatenate(naives).ravel()

def r2(y, p):
    return 1 - ((y-p)**2).sum() / ((y-y.mean())**2).sum()

import json
pretrain_metrics = {
    "stage": "pretrain", "datasets": dataset_names,
    "model_r2": round(float(r2(act, pred)), 4),
    "model_mae": round(float(np.abs(act-pred).mean()), 3),
    "naive_r2": round(float(r2(act, nai)), 4),
    "naive_mae": round(float(np.abs(act-nai).mean()), 3),
    "n_entities": len(entities), "n_valid_windows": int(len(np.concatenate(actuals))),
}
json.dump(pretrain_metrics, open("/kaggle/working/pretrain_metrics.json", "w"), indent=2)
print(json.dumps(pretrain_metrics, indent=2))

## Stage 2: finetune on real paratransit demand (AAR)

Requires the optional Kaggle Dataset with `aar_hourly_borough.csv.gz` and `weather_daily.csv` attached (see EXTRA_PANELS_DIR above). If not present, this stage prints instructions and stops rather than silently skipping.

Transfer strategy: extend the entity and dataset embedding tables with 5 new borough entities and one new `aar` dataset id (randomly initialized), train those plus the two heads at the full learning rate, and train the pretrained transformer backbone at `BACKBONE_LR_MULT` (5%) of that rate so it adapts without forgetting what it learned from 2B+ mass-transit events.

In [ ]:
aar_path = extra / "aar_hourly_borough.csv.gz"
if not aar_path.exists():
    print(f"AAR panel not found at {aar_path}.")
    print("To run Stage 2: create a Kaggle Dataset named "
          "'transitpulse-real-panels' from the repo's realdata/panels/ "
          "folder (aar_hourly_borough.csv.gz + weather_daily.csv at minimum), "
          "attach it to this notebook, and re-run from this cell.")
else:
    aar_raw = pd.read_csv(aar_path)
    aar = (aar_raw.groupby(["day", "hour", "borough"]).trips.sum()
           .rename("y").reset_index().rename(columns={"day": "date"}))
    aar["zone"] = aar["borough"]
    aar["dataset"] = "aar"
    aar["date"] = pd.to_datetime(aar["date"])
    days = pd.date_range(aar.date.min(), aar.date.max(), freq="D")
    grid = pd.MultiIndex.from_product(
        [days, range(24), aar.zone.unique()],
        names=["date", "hour", "zone"]).to_frame(index=False)
    aar_full = grid.merge(aar[["date","hour","zone","y"]], how="left",
                          on=["date","hour","zone"])
    aar_full["y"] = aar_full.y.fillna(0)
    aar_full["dataset"] = "aar"
    print(f"AAR finetune panel: {len(aar_full):,} rows, "
          f"{aar_full.y.sum():,.0f} bookings, {aar_full.zone.nunique()} boroughs")

In [ ]:
if aar_path.exists():
    M_ft, CAL_ft, entities_ft, e_index_ft, dataset_of_ft, clock_ft = build_matrix(aar_full)
    dataset_names_ft = dataset_names + ["aar"]
    ds_index_ft = {d: i for i, d in enumerate(dataset_names_ft)}
    entity_dataset_id_ft = np.array([ds_index_ft["aar"] for _ in entities_ft])

    # Expand the pretrained model's embedding tables to fit the new
    # entities/dataset, copying pretrained weights for anything reused
    # and randomly initializing only the genuinely new rows.
    ft_model = DemandTransformer(len(entities_ft), len(dataset_names_ft), len(QUANTILES)).to(dev)
    pretrained_state = torch.load("/kaggle/working/pretrained_backbone.pt")
    ft_state = ft_model.state_dict()
    for k, v in pretrained_state.items():
        if k in ("entity_emb.weight", "dataset_emb.weight"):
            continue  # new vocab size, keep fresh init
        ft_state[k] = v
    ft_model.load_state_dict(ft_state)

    # global M/CAL/entity_dataset_id must point at the finetune matrices
    # for the Windows dataset class defined above to work unchanged
    M, CAL, entities, entity_dataset_id = M_ft, CAL_ft, entities_ft, entity_dataset_id_ft

    valid_start_ft = len(clock_ft) - 90 * 24
    starts_ft = np.arange(0, len(clock_ft) - INPUT_HOURS - HORIZON, 24)
    train_starts_ft = starts_ft[starts_ft + INPUT_HOURS + HORIZON <= valid_start_ft]
    valid_starts_ft = starts_ft[starts_ft + INPUT_HOURS >= valid_start_ft]

    ft_train_dl = DataLoader(Windows(train_starts_ft), batch_size=64, shuffle=True)
    ft_valid_dl = DataLoader(Windows(valid_starts_ft), batch_size=64)

    backbone_params, new_params = [], []
    for name, p in ft_model.named_parameters():
        (new_params if name.startswith(("entity_emb", "dataset_emb",
                                        "mean_head", "quantile_head"))
         else backbone_params).append(p)
    ft_opt = torch.optim.AdamW([
        {"params": backbone_params, "lr": FINETUNE_LR * BACKBONE_LR_MULT},
        {"params": new_params, "lr": FINETUNE_LR},
    ], weight_decay=1e-4)

    best_ft = float("inf")
    for ep in range(FINETUNE_EPOCHS):
        ft_model.train(); t0 = time.time(); tot = 0.0
        for feats, e_id, d_id, y in ft_train_dl:
            feats, e_id, d_id, y = feats.to(dev), e_id.to(dev), d_id.to(dev), y.to(dev)
            ft_opt.zero_grad(set_to_none=True)
            mean_rate, q = ft_model(feats, e_id, d_id)
            loss = nll(mean_rate, y) + 0.5 * pinball_loss(q, y, QUANTILES)
            loss.backward(); ft_opt.step()
            tot += loss.item() * len(e_id)
        ft_model.eval(); vtot, n = 0.0, 0
        with torch.no_grad():
            for feats, e_id, d_id, y in ft_valid_dl:
                feats, e_id, d_id, y = feats.to(dev), e_id.to(dev), d_id.to(dev), y.to(dev)
                mean_rate, q = ft_model(feats, e_id, d_id)
                vtot += (nll(mean_rate, y) + 0.5*pinball_loss(q, y, QUANTILES)).item() * len(e_id)
                n += len(e_id)
        v = vtot / n
        print(f"finetune epoch {ep}: train {tot/len(ft_train_dl.dataset):.4f} "
              f"valid {v:.4f} ({time.time()-t0:.0f}s)", flush=True)
        if v < best_ft:
            best_ft = v
            torch.save(ft_model.state_dict(), "/kaggle/working/finetuned_aar_model.pt")
else:
    print("skipping Stage 2 (no AAR panel attached)")

In [ ]:
if aar_path.exists():
    ft_model.load_state_dict(torch.load("/kaggle/working/finetuned_aar_model.pt"))
    ft_model.eval()
    preds, actuals, naives = [], [], []
    with torch.no_grad():
        for feats, e_id, d_id, y in ft_valid_dl:
            mean_rate, _ = ft_model(feats.to(dev), e_id.to(dev), d_id.to(dev))
            preds.append(mean_rate.cpu().numpy()); actuals.append(y.numpy())
            x = np.expm1(feats[:, :, 0].numpy())
            naives.append(x[:, INPUT_HOURS-168:INPUT_HOURS-168+HORIZON])
    pred = np.concatenate(preds).ravel(); act = np.concatenate(actuals).ravel()
    nai = np.concatenate(naives).ravel()

    finetune_metrics = {
        "stage": "finetune_aar", "pretrained_on": dataset_names,
        "model_r2": round(float(r2(act, pred)), 4),
        "model_mae": round(float(np.abs(act-pred).mean()), 3),
        "naive_r2": round(float(r2(act, nai)), 4),
        "naive_mae": round(float(np.abs(act-nai).mean()), 3),
    }
    json.dump(finetune_metrics, open("/kaggle/working/finetune_metrics.json", "w"), indent=2)
    print(json.dumps(finetune_metrics, indent=2))
    print()
    print("Compare this R2/MAE against realdata/metrics_real.json's XGBoost "
          "result on the same AAR data (R2 0.987 vs 0.923 naive, 90-day "
          "holdout) before claiming the transformer is better. Report both, "
          "always, whichever wins.")

Download `pretrained_backbone.pt`, `finetuned_aar_model.pt` (if Stage 2 ran), `pretrain_metrics.json`, and `finetune_metrics.json` from the notebook output and commit the metrics JSONs to `kaggle/results/`. Never quote a model number without the naive-baseline number next to it, and never quote the transformer's AAR result without also citing realdata/metrics_real.json's XGBoost result on the identical data and holdout window — the honest comparison is the point, not just the bigger model.